# 470 套数据 · 条件 CVAE · 生成与量化评估（训练优化 v1）

基于 `260523-74条件生成与量化评估（MVP）.ipynb` 副本，面向 `data/processed/**/house_*.json` 全量数据。

| 优化项 | 说明 |
|---|---|
| 递归数据扫描 | `data/processed` 三批子目录 |
| 持久化划分 | `train_val_split_v470.json`，训练与 Step 7 共用 |
| 拓扑加噪 | 训练时边 dropout + 节点坐标微扰 |
| KL Annealing | β 从 0 线性升至 `KL_WEIGHT_MAX` |
| 梯度积累 + AMP | 有效 batch 更大、显存更省 |
| 合成拓扑探针 | 每 `PROBE_EVERY` epoch 监控非空体素率 |
| Best-of-K 评估 | Step 7 条件生成多次采样取最优 |
| **断点续训** | 每 epoch 存 Drive 检查点，Colab 断连后 Step 4 自动恢复 |

| 步骤 | 作用 |
|---|---|
| Step 0–4 | 安装依赖、配置、预处理、建模、训练 |
| **Step 1b** | 合成拓扑工具（训练探针 + Step 6 共用） |
| **Step R** | 加载已训练权重 |
| Step 5–7 | 重建评估、条件生成、量化对比（**Step 7 不需跑 Step 6**） |
| Step 8 | 交互式生成 |

> **未纳入本版**：FiLM、Attention Pooling、LR Warmup（留作 v2 架构实验）


In [ ]:

# Step 0: 安装依赖（Colab 首次运行）
!pip -q install torch_geometric


In [ ]:
# Step 1: 全局配置 + 数据工具
import os, json, random, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader as GraphDataLoader
from torch_geometric.nn import HeteroConv, SAGEConv, GATv2Conv, global_mean_pool

USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        USE_DRIVE = False

def resolve_data_dir():
    candidates = [
        '/content/drive/MyDrive/master_thesis/data',
        '/content/drive/MyDrive/260509_dataset',
        '/content/data',
        os.path.join(os.getcwd(), 'data'),
        os.path.abspath(os.path.join(os.getcwd(), '..', 'data')),
    ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    return candidates[0]

DATA_DIR = resolve_data_dir()


VOXEL_SIZE = 300.0
RES_X, RES_Y, RES_Z = 64, 128, 32

CHANNEL_MAP = {
    'empty': 0, 'entryway': 1, 'living_room': 2, 'dining_room': 3,
    'kitchen': 4, 'bedroom': 5, 'bathroom': 6, 'corridor': 7,
    'stairs': 8, 'utility': 9, 'balcony': 10, 'multi_purpose': 11,
}
ROOM_TYPES = [k for k in CHANNEL_MAP if k != 'empty']
NUM_CHANNELS = len(CHANNEL_MAP)
NODE_IN_DIM = 8
COND_DIM = 3 + len(ROOM_TYPES) + 3

LIGHTING_ACCESS_MAP = {'direct': 1.0, 'indirect': 0.5, 'none': 0.0}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device}')
print(f'数据目录: {DATA_DIR}  存在: {os.path.isdir(DATA_DIR)}')
print(f'节点维={NODE_IN_DIM}  条件维={COND_DIM}')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_node_supertype(room_type):
    if room_type in ['bedroom', 'living_room', 'dining_room', 'multi_purpose']:
        return 'living'
    if room_type in ['kitchen', 'bathroom', 'utility', 'balcony']:
        return 'service'
    return 'circulation'


def check_hetero_adjacency(r1, r2, tol=150, min_shared_len=300):
    if r1.get('floor', 1) != r2.get('floor', 1):
        if r1['type'] == 'stairs' or r2['type'] == 'stairs':
            ox = max(0, min(r1['box_max'][0], r2['box_max'][0]) - max(r1['box_min'][0], r2['box_min'][0]))
            oy = max(0, min(r1['box_max'][1], r2['box_max'][1]) - max(r1['box_min'][1], r2['box_min'][1]))
            if ox > 0 and oy > 0:
                return 'vertical'
        return None
    if r1.get('floor', 1) == r2.get('floor', 1):
        ox = max(0, min(r1['box_max'][0], r2['box_max'][0]) - max(r1['box_min'][0], r2['box_min'][0]) + tol)
        oy = max(0, min(r1['box_max'][1], r2['box_max'][1]) - max(r1['box_min'][1], r2['box_min'][1]) + tol)
        if (ox > min_shared_len and oy > 0) or (oy > min_shared_len and ox > 0):
            return 'horizontal'
    return None


def effective_lighting_ratio(room):
    dx = room['box_max'][0] - room['box_min'][0]
    dy = room['box_max'][1] - room['box_min'][1]
    floor_area = max(dx * dy, 1.0)
    eff = room.get('effective_lighting', [])
    total = sum(float(e.get('area', 0)) * float(e.get('attenuation', 1.0)) for e in eff)
    return min(total / floor_area, 5.0) / 5.0


def build_room_features(room, build_min, b_size):
    dx = room['box_max'][0] - room['box_min'][0]
    dy = room['box_max'][1] - room['box_min'][1]
    area = (dx * dy) / 1e6
    aspect = max(dx, dy) / max(min(dx, dy), 1.0)
    cx = ((room['box_min'][0] + room['box_max'][0]) / 2 - build_min[0]) / (b_size[0] + 1e-5)
    cy = ((room['box_min'][1] + room['box_max'][1]) / 2 - build_min[1]) / (b_size[1] + 1e-5)
    cz = ((room['box_min'][2] + room['box_max'][2]) / 2 - build_min[2]) / (b_size[2] + 1e-5)
    lighting_access = LIGHTING_ACCESS_MAP.get(room.get('lighting_access', 'none'), 0.0)
    lighting_priority = float(room.get('lighting_priority', 0)) / 10.0
    lighting_ratio = effective_lighting_ratio(room)
    return [area, aspect, cx, cy, cz, lighting_access, lighting_priority, lighting_ratio]


def build_condition_vector(data):
    meta = data.get('metadata', {})
    stats = meta.get('stats', {})
    bsize = meta.get('building_size', {'x': 1.0, 'y': 1.0, 'z': 1.0})
    rooms = data.get('rooms', [])
    total = max(len(rooms), 1)

    cond = [
        float(bsize.get('x', 1.0)) / 30000.0,
        float(bsize.get('y', 1.0)) / 30000.0,
        float(bsize.get('z', 1.0)) / 9000.0,
    ]
    for rt in ROOM_TYPES:
        cond.append(float(stats.get(rt, 0)) / total)

    direct = indirect = none = 0
    for r in rooms:
        acc = r.get('lighting_access', 'none')
        if acc == 'direct':
            direct += 1
        elif acc == 'indirect':
            indirect += 1
        else:
            none += 1
    cond.extend([direct / total, indirect / total, none / total])
    return cond


def json_to_sample(data):
    rooms = data.get('rooms', [])
    if not rooms:
        return None

    all_coords = np.array([r['box_min'] for r in rooms] + [r['box_max'] for r in rooms])
    build_min = all_coords.min(axis=0)
    build_max = all_coords.max(axis=0)
    b_size = build_max - build_min

    hetero = HeteroData()
    nodes_dict = {'living': [], 'service': [], 'circulation': []}
    id_to_idx = {}

    for r in rooms:
        ntype = get_node_supertype(r['type'])
        id_to_idx[r['id']] = (ntype, len(nodes_dict[ntype]))
        nodes_dict[ntype].append(build_room_features(r, build_min, b_size))

    for ntype, feats in nodes_dict.items():
        hetero[ntype].x = torch.tensor(feats, dtype=torch.float32) if feats else torch.empty((0, NODE_IN_DIM))

    edges_dict = {}
    for i in range(len(rooms)):
        for j in range(i + 1, len(rooms)):
            r1, r2 = rooms[i], rooms[j]
            rel = check_hetero_adjacency(r1, r2)
            if not rel:
                continue
            t1, idx1 = id_to_idx[r1['id']]
            t2, idx2 = id_to_idx[r2['id']]
            for src_t, dst_t, si, di in ((t1, t2, idx1, idx2), (t2, t1, idx2, idx1)):
                key = (src_t, rel, dst_t)
                edges_dict.setdefault(key, []).append([si, di])

    for key, elist in edges_dict.items():
        hetero[key].edge_index = torch.tensor(elist, dtype=torch.long).t().contiguous()

    grid = np.zeros((RES_X, RES_Y, RES_Z), dtype=np.int8)
    phys_center_xy = (build_min[:2] + build_max[:2]) / 2.0
    offset_xy = np.array([RES_X * VOXEL_SIZE / 2, RES_Y * VOXEL_SIZE / 2]) - phys_center_xy
    z_min_phys = build_min[2]

    for r in rooms:
        ch = CHANNEL_MAP.get(r['type'], 0)
        ix_min = int((r['box_min'][0] + offset_xy[0]) / VOXEL_SIZE)
        ix_max = int((r['box_max'][0] + offset_xy[0]) / VOXEL_SIZE)
        iy_min = int((r['box_min'][1] + offset_xy[1]) / VOXEL_SIZE)
        iy_max = int((r['box_max'][1] + offset_xy[1]) / VOXEL_SIZE)
        iz_min = int((r['box_min'][2] - z_min_phys) / VOXEL_SIZE)
        iz_max = int((r['box_max'][2] - z_min_phys) / VOXEL_SIZE)
        grid[max(0, ix_min):min(RES_X, ix_max), max(0, iy_min):min(RES_Y, iy_max), max(0, iz_min):min(RES_Z, iz_max)] = ch

    one_hot = np.zeros((NUM_CHANNELS, RES_X, RES_Y, RES_Z), dtype=np.float32)
    for c in range(NUM_CHANNELS):
        one_hot[c] = (grid == c).astype(np.float32)

    cond = torch.tensor(build_condition_vector(data), dtype=torch.float32)
    return {'graph': hetero, 'voxel': torch.tensor(one_hot, dtype=torch.float32), 'condition': cond}


def dice_loss_with_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (0, 2, 3, 4)
    inter = (probs * targets).sum(dims)
    union = probs.sum(dims) + targets.sum(dims)
    dice = (2 * inter + eps) / (union + eps)
    return 1.0 - dice.mean()


def graph_batch_size(batch):
    return int(getattr(batch, 'num_graphs', 1))


def graph_batch_dict(batch):
    if hasattr(batch, 'batch_dict'):
        return batch.batch_dict
    bd = {}
    for ntype in batch.node_types:
        if batch[ntype].x.size(0) > 0:
            if hasattr(batch[ntype], 'batch'):
                bd[ntype] = batch[ntype].batch
            else:
                bd[ntype] = torch.zeros(
                    batch[ntype].x.size(0), dtype=torch.long, device=batch[ntype].x.device
                )
    return bd


def graph_condition(batch):
    bs = graph_batch_size(batch)
    if not hasattr(batch, 'condition'):
        raise AttributeError('batch 缺少 condition：请用 prepare_graph_batch(hg, condition=tensor)')
    cond = batch.condition
    if cond.dim() == 1:
        cond = cond.unsqueeze(0)
    return cond.view(bs, -1)


def prepare_graph_batch(hg, condition=None):
    """将 condition 挂到 HeteroData 并确保 Batch 后仍可读取。"""
    if condition is not None:
        cond = condition.unsqueeze(0) if condition.dim() == 1 else condition
        hg.condition = cond
    batch = next(iter(GraphDataLoader([hg], batch_size=1)))
    if condition is not None and not hasattr(batch, 'condition'):
        batch.condition = hg.condition
    return batch


def forward_model(model, batch, condition=None):
    """显式传入 condition，避免 HeteroDataBatch 丢字段。"""
    cond = condition if condition is not None else graph_condition(batch)
    if cond.dim() == 1:
        cond = cond.unsqueeze(0)
    mu, logvar = model.encoder(
        batch.x_dict, batch.edge_index_dict, graph_batch_dict(batch)
    )
    z = model.reparameterize(mu, logvar)
    logits = model.decoder(z, cond.to(z.device))
    return logits, mu, logvar

from datetime import datetime

TOTAL_JSON_COUNT = 0  # 在下方 v1 配置块中重新统计
TRAIN_SAMPLE_COUNT = None
TRAIN_TIMESTAMP = None
EXPERIMENT_META = {}


def set_experiment_meta(train_count, total_count=None, timestamp=None, extra=None):
    global TRAIN_SAMPLE_COUNT, TRAIN_TIMESTAMP, EXPERIMENT_META, TOTAL_JSON_COUNT
    TRAIN_SAMPLE_COUNT = int(train_count)
    if total_count is not None:
        TOTAL_JSON_COUNT = int(total_count)
    TRAIN_TIMESTAMP = timestamp or datetime.now().strftime('%Y%m%d_%H%M%S')
    EXPERIMENT_META = {
        'train_samples': TRAIN_SAMPLE_COUNT,
        'total_json': TOTAL_JSON_COUNT,
        'trained_at': TRAIN_TIMESTAMP,
    }
    if extra:
        EXPERIMENT_META.update(extra)
    return EXPERIMENT_META


def experiment_report_name(prefix, ext='csv'):
    ts = TRAIN_TIMESTAMP or datetime.now().strftime('%Y%m%d_%H%M%S')
    n_train = TRAIN_SAMPLE_COUNT if TRAIN_SAMPLE_COUNT is not None else 'NA'
    n_total = TOTAL_JSON_COUNT if TOTAL_JSON_COUNT is not None else 'NA'
    return f"{prefix}_json{n_total}_train{n_train}_{ts}.{ext}"


def experiment_report_path(prefix, subdir='eval_reports', ext='csv'):
    out = os.path.join(DATA_DIR, subdir)
    os.makedirs(out, exist_ok=True)
    return os.path.join(out, experiment_report_name(prefix, ext))

# ---------- 470 训练优化 v1 配置 ----------
JSON_ROOT = os.path.join(DATA_DIR, 'processed')
OUT_DIR = os.path.join(DATA_DIR, 'processed_tensors_v470')
# Drive 空间不足时可改 Colab 本地（~6GB，会话结束丢失，需重跑 Step 2）：
# OUT_DIR = '/content/processed_tensors_v470'
CACHE_VERSION = 'v470_train_opt_v1'
WEIGHT_FILENAME = 'spatial_modal_cvae_470_v1.pth'
SPLIT_FILENAME = 'train_val_split_v470.json'

# 训练超参（Step 4 读取）
BATCH_SIZE = 2
ACCUM_STEPS = 4          # 有效 batch ≈ BATCH_SIZE * ACCUM_STEPS
USE_AMP = True
EPOCHS = 120
PATIENCE = 20
LR = 1e-3
WEIGHT_DECAY = 1e-5
DICE_WEIGHT = 0.3
KL_WEIGHT_MAX = 1e-4
KL_ANNEAL_EPOCHS = 25
EDGE_DROP_PROB = 0.10
NODE_JITTER = 0.02
AUGMENT_PROB = 0.5       # 每个训练样本施加拓扑加噪的概率
PROBE_EVERY = 10         # 合成拓扑随堂测试间隔（epoch）
GEN_EVAL_K = 8           # Step 7 条件生成 best-of-K

# 断点续训（Colab 断连恢复）
CHECKPOINT_FILENAME = 'spatial_modal_cvae_470_v1_checkpoint.pt'
RESUME_IF_CHECKPOINT = True   # 有检查点则自动续训
FORCE_NEW_TRAINING = False    # True = 忽略检查点，强制从头训
MANUAL_RESUME_EPOCH = 0       # 无检查点时用：填上次完成的 epoch（如 40）
LOG_EVERY = 5                 # 每 N epoch 打印一行进度

VAL_RATIO = 0.2
SPLIT_SEED = 42


def discover_json_files(root=None):
    root = root or JSON_ROOT
    if not os.path.isdir(root):
        return []
    return sorted(
        f for f in glob.glob(os.path.join(root, '**', 'house_*.json'), recursive=True)
        if not f.endswith('_topology.json')
    )


def split_path():
    return os.path.join(DATA_DIR, SPLIT_FILENAME)


def weight_path():
    return os.path.join(DATA_DIR, WEIGHT_FILENAME)


def checkpoint_path():
    return os.path.join(DATA_DIR, CHECKPOINT_FILENAME)


def load_train_val_split(pt_names, seed=SPLIT_SEED, val_ratio=VAL_RATIO):
    path = split_path()
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            saved = json.load(f)
        train_names = saved.get('train', [])
        val_names = saved.get('val', [])
        known = set(pt_names)
        train_names = [n for n in train_names if n in known]
        val_names = [n for n in val_names if n in known]
        if train_names or val_names:
            return train_names, val_names, saved
    rng = random.Random(seed)
    names = list(pt_names)
    rng.shuffle(names)
    n_val = max(1, int(len(names) * val_ratio))
    val_names = names[-n_val:]
    train_names = names[:-n_val]
    meta = {
        'seed': seed,
        'val_ratio': val_ratio,
        'total': len(names),
        'train_count': len(train_names),
        'val_count': len(val_names),
        'cache_version': CACHE_VERSION,
    }
    with open(path, 'w', encoding='utf-8') as f:
        json.dump({'train': train_names, 'val': val_names, 'meta': meta}, f, indent=2, ensure_ascii=False)
    return train_names, val_names, meta


def resolve_source_json(sample, pt_basename=None):
    if isinstance(sample, dict) and sample.get('source_json') and os.path.exists(sample['source_json']):
        return sample['source_json']
    if pt_basename:
        cand = os.path.join(DATA_DIR, pt_basename.replace('.pt', '.json'))
        if os.path.exists(cand):
            return cand
    return None


def augment_batch_topology(batch, edge_drop_prob=EDGE_DROP_PROB, node_jitter=NODE_JITTER):
    """直接在 HeteroDataBatch 上加噪，避免 to_data_list 拆分 batch 元数据出错。"""
    for edge_key in batch.edge_types:
        store = batch[edge_key]
        ei = store.edge_index
        if ei.numel() == 0:
            continue
        keep = torch.rand(ei.size(1), device=ei.device) > edge_drop_prob
        if not keep.any():
            keep = torch.zeros(ei.size(1), dtype=torch.bool, device=ei.device)
            keep[0] = True
        store.edge_index = ei[:, keep]
    for ntype in batch.node_types:
        x = batch[ntype].x
        if x is None or x.numel() == 0:
            continue
        noise = torch.zeros_like(x)
        if x.size(1) >= 5:
            noise[:, 2:5] = torch.randn(x.size(0), 3, device=x.device) * node_jitter
        batch[ntype].x = (x + noise).clamp(0.0, 5.0)
    return batch


# 覆盖 TOTAL_JSON_COUNT（递归统计）
TOTAL_JSON_COUNT = len(discover_json_files())
print(f'JSON 根目录: {JSON_ROOT}  发现 JSON: {TOTAL_JSON_COUNT}')
print(f'缓存目录: {OUT_DIR}  权重: {WEIGHT_FILENAME}')


In [ ]:
# Step 1b: 合成拓扑工具（训练探针 & Step 6 条件生成共用）
import math
import networkx as nx

DEFAULT_ROOM_SIZE = {
    'entryway': (2400, 2400, 3000),
    'living_room': (6000, 4500, 3000),
    'dining_room': (3600, 3300, 3000),
    'kitchen': (3300, 3000, 3000),
    'bedroom': (3600, 3600, 3000),
    'bathroom': (2400, 2400, 3000),
    'corridor': (1800, 2400, 3000),
    'stairs': (3000, 3000, 6000),
    'utility': (2400, 2400, 3000),
    'balcony': (3000, 1800, 3000),
    'multi_purpose': (3600, 3300, 3000),
}

SYNTHETIC_PROBE_SPECS = [
    {'name': 'compact_2f', 'site_x': 15000, 'site_y': 12000, 'seed': 11,
     'room_counts': {'entryway': 1, 'living_room': 1, 'dining_room': 1, 'kitchen': 1,
                     'bedroom': 2, 'bathroom': 1, 'corridor': 1, 'stairs': 1}},
    {'name': 'standard_3br', 'site_x': 18000, 'site_y': 15000, 'seed': 22,
     'room_counts': {'entryway': 1, 'living_room': 1, 'dining_room': 1, 'kitchen': 1,
                     'bedroom': 3, 'bathroom': 2, 'corridor': 2, 'stairs': 1, 'balcony': 1}},
    {'name': 'large_4br', 'site_x': 21000, 'site_y': 18000, 'seed': 33,
     'room_counts': {'entryway': 1, 'living_room': 1, 'dining_room': 1, 'kitchen': 1,
                     'bedroom': 4, 'bathroom': 2, 'corridor': 2, 'stairs': 1, 'balcony': 1, 'utility': 1}},
]


def snap_modulus(v):
    return round(float(v) / VOXEL_SIZE) * VOXEL_SIZE


def build_user_request(site_x, site_y, room_counts, site_z=6000):
    counts = {k: int(v) for k, v in room_counts.items() if int(v) > 0}
    return {
        'site_x': float(site_x), 'site_y': float(site_y), 'site_z': float(site_z),
        'room_counts': counts,
    }


def build_program_topology(room_counts, seed=42):
    G = nx.Graph()
    nodes = []
    bath_i = corr_i = 0
    for r_type, count in room_counts.items():
        for i in range(count):
            nid = f"{r_type}_{i}"
            if r_type in ['entryway', 'living_room', 'dining_room', 'kitchen']:
                floor = 1
            elif r_type in ['bedroom', 'balcony']:
                floor = 2
            elif r_type == 'bathroom':
                floor = 1 if bath_i % 2 == 0 else 2
                bath_i += 1
            elif r_type == 'corridor':
                floor = 1 if corr_i % 2 == 0 else 2
                corr_i += 1
            elif r_type == 'stairs':
                floor = '1&2'
            else:
                floor = 1
            nodes.append((nid, r_type, floor))
            G.add_node(nid, type=r_type, floor=floor)

    f1_corr = [n for n, t, f in nodes if t == 'corridor' and f == 1]
    f2_corr = [n for n, t, f in nodes if t == 'corridor' and f == 2]
    f1_hub = f1_corr[0] if f1_corr else next((n for n, t, f in nodes if t == 'living_room'), nodes[0][0])
    f2_hub = f2_corr[0] if f2_corr else next((n for n, t, f in nodes if t == 'bedroom'), nodes[-1][0])

    edge_types = {}
    for nid, r_type, floor in nodes:
        if r_type == 'stairs':
            continue
        hub = f1_hub if floor == 1 else f2_hub
        if nid != hub:
            G.add_edge(nid, hub)
            edge_types[(nid, hub)] = 'horizontal'
            edge_types[(hub, nid)] = 'horizontal'

    stairs = [n for n, t, f in nodes if t == 'stairs']
    if stairs:
        st = stairs[0]
        if f1_hub != st:
            G.add_edge(st, f1_hub)
            edge_types[(st, f1_hub)] = 'vertical'
        if f2_hub != st and f2_hub != f1_hub:
            G.add_edge(st, f2_hub)
            edge_types[(st, f2_hub)] = 'vertical'

    pos = nx.spring_layout(G, seed=seed, k=1.2)
    return G, pos, nodes, edge_types


def layout_rooms_from_program(user_req, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    G, pos, nodes, edge_types = build_program_topology(user_req['room_counts'], seed=seed)
    sx, sy = user_req['site_x'], user_req['site_y']
    rooms = []
    for nid, r_type, floor in nodes:
        w, d, h = DEFAULT_ROOM_SIZE.get(r_type, (3600, 3600, 3000))
        px, py = pos[nid]
        cx = (px + 1) / 2 * (sx * 0.7) + sx * 0.15
        cy = (py + 1) / 2 * (sy * 0.7) + sy * 0.15
        cx, cy = snap_modulus(cx), snap_modulus(cy)
        w, d = snap_modulus(w), snap_modulus(d)
        z0 = 0 if floor == 1 or floor == '1&2' else 3000
        z1 = z0 + h
        rooms.append({
            'id': nid, 'type': r_type, 'floor': 1 if floor == '1&2' else floor,
            'box_min': [max(0, cx - w / 2), max(0, cy - d / 2), z0],
            'box_max': [min(sx, cx + w / 2), min(sy, cy + d / 2), z1],
            'lighting_access': 'direct' if r_type in ['living_room', 'bedroom', 'balcony'] else 'indirect',
            'lighting_priority': 8 if r_type in ['living_room', 'bedroom'] else 4,
            'effective_lighting': [],
        })
    return rooms, G, pos, edge_types


def request_to_house_json(user_req, rooms):
    stats = {t: 0 for t in ROOM_TYPES}
    for r in rooms:
        stats[r['type']] = stats.get(r['type'], 0) + 1
    return {
        'metadata': {
            'stats': stats,
            'total_rooms': len(rooms),
            'building_size': {'x': user_req['site_x'], 'y': user_req['site_y'], 'z': user_req['site_z']},
            'constraints': {'modulus': int(VOXEL_SIZE)},
        },
        'rooms': rooms,
    }


@torch.no_grad()
def synthetic_graph_forward(user_req, model, seed=42):
    rooms, _, _, _ = layout_rooms_from_program(user_req, seed=seed)
    data = request_to_house_json(user_req, rooms)
    sample = json_to_sample(data)
    if sample is None:
        return 0, 0.0
    hg = sample['graph']
    batch = prepare_graph_batch(hg, condition=sample['condition']).to(device)
    model.eval()
    logits, _, _ = forward_model(model, batch, sample['condition'])
    pred = torch.argmax(logits[0], dim=0).cpu().numpy()
    n_occ = int((pred > 0).sum())
    occ_ratio = float(n_occ / max(pred.size, 1))
    return n_occ, occ_ratio


@torch.no_grad()
def run_synthetic_probe_eval(model, specs=None, verbose=True):
    specs = specs or SYNTHETIC_PROBE_SPECS
    results = []
    for spec in specs:
        req = build_user_request(spec['site_x'], spec['site_y'], spec['room_counts'])
        n_occ, occ_ratio = synthetic_graph_forward(req, model, seed=spec.get('seed', 42))
        row = {'name': spec['name'], 'n_occ': n_occ, 'occ_ratio': occ_ratio}
        results.append(row)
        if verbose:
            print(f"  [{spec['name']}] 非空体素={n_occ} ({occ_ratio*100:.4f}%)")
    mean_occ = float(np.mean([r['n_occ'] for r in results]))
    hit = sum(1 for r in results if r['n_occ'] > 0)
    return {'per_probe': results, 'mean_n_occ': mean_occ, 'nonempty_hits': hit, 'total': len(results)}


@torch.no_grad()
def generate_voxels_from_request(user_req, model, seed=42):
    rooms, G, pos, edge_types = layout_rooms_from_program(user_req, seed=seed)
    data = request_to_house_json(user_req, rooms)
    sample = json_to_sample(data)
    hg = sample['graph']
    batch = prepare_graph_batch(hg, condition=sample['condition']).to(device)
    model.eval()
    logits, _, _ = forward_model(model, batch, sample['condition'])
    pred_cls = torch.argmax(logits[0], dim=0).cpu().numpy()
    return pred_cls, sample, rooms, G, pos


def count_components_3d(mask):
    if not mask.any():
        return 0
    visited = np.zeros(mask.shape, dtype=bool)
    n_comp = 0
    xs, ys, zs = np.where(mask)
    for x, y, z in zip(xs, ys, zs):
        if visited[x, y, z]:
            continue
        n_comp += 1
        stack = [(int(x), int(y), int(z))]
        visited[x, y, z] = True
        while stack:
            cx, cy, cz = stack.pop()
            for dx, dy, dz in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)):
                nx, ny, nz = cx+dx, cy+dy, cz+dz
                if 0 <= nx < mask.shape[0] and 0 <= ny < mask.shape[1] and 0 <= nz < mask.shape[2]:
                    if mask[nx, ny, nz] and not visited[nx, ny, nz]:
                        visited[nx, ny, nz] = True
                        stack.append((nx, ny, nz))
    return n_comp


def voxel_grid_phys_offset(rooms):
    """与 json_to_sample 编码一致的 XY/Z 偏移，用于体素→物理坐标反变换。"""
    if not rooms:
        return np.array([0.0, 0.0]), 0.0
    all_coords = np.array([r['box_min'] for r in rooms] + [r['box_max'] for r in rooms])
    build_min = all_coords.min(axis=0)
    build_max = all_coords.max(axis=0)
    phys_center_xy = (build_min[:2] + build_max[:2]) / 2.0
    offset_xy = np.array([RES_X * VOXEL_SIZE / 2, RES_Y * VOXEL_SIZE / 2]) - phys_center_xy
    return offset_xy, float(build_min[2])


def voxels_to_boxes(pred_cls, user_req=None, rooms=None):
    TYPE_NAMES_LOCAL = {v: k for k, v in CHANNEL_MAP.items() if v > 0}
    offset_xy, z_min_phys = voxel_grid_phys_offset(rooms) if rooms else (np.array([0.0, 0.0]), 0.0)
    boxes = []
    for cid in range(1, NUM_CHANNELS):
        mask = pred_cls == cid
        if not mask.any():
            continue
        visited = np.zeros(mask.shape, dtype=bool)
        xs, ys, zs = np.where(mask)
        for x, y, z in zip(xs, ys, zs):
            if visited[x, y, z]:
                continue
            stack = [(int(x), int(y), int(z))]
            comp = []
            visited[x, y, z] = True
            while stack:
                cx, cy, cz = stack.pop()
                comp.append((cx, cy, cz))
                for dx, dy, dz in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)):
                    nx, ny, nz = cx+dx, cy+dy, cz+dz
                    if 0 <= nx < mask.shape[0] and 0 <= ny < mask.shape[1] and 0 <= nz < mask.shape[2]:
                        if mask[nx, ny, nz] and not visited[nx, ny, nz]:
                            visited[nx, ny, nz] = True
                            stack.append((nx, ny, nz))
            arr = np.array(comp)
            ix0, iy0, iz0 = arr.min(axis=0)
            ix1, iy1, iz1 = arr.max(axis=0) + 1
            rtype = TYPE_NAMES_LOCAL[cid]
            boxes.append({
                'type': rtype,
                'box_min': [
                    snap_modulus(ix0 * VOXEL_SIZE - offset_xy[0]),
                    snap_modulus(iy0 * VOXEL_SIZE - offset_xy[1]),
                    snap_modulus(iz0 * VOXEL_SIZE + z_min_phys),
                ],
                'box_max': [
                    snap_modulus(ix1 * VOXEL_SIZE - offset_xy[0]),
                    snap_modulus(iy1 * VOXEL_SIZE - offset_xy[1]),
                    snap_modulus(iz1 * VOXEL_SIZE + z_min_phys),
                ],
            })
    return boxes


print('Step 1b 就绪: 合成拓扑 + 条件生成评估函数 +', len(SYNTHETIC_PROBE_SPECS), '个探针')


In [ ]:
# Step 2: 批量预处理 JSON -> .pt（递归扫描 processed/ 子目录）
os.makedirs(OUT_DIR, exist_ok=True)
meta_path = os.path.join(OUT_DIR, '_cache_meta.json')

json_files = discover_json_files()
if not json_files:
    raise FileNotFoundError(f'未找到 JSON: {JSON_ROOT}（请确认 data/processed 已上传）')

need_rebuild = True
if os.path.exists(meta_path):
    with open(meta_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    need_rebuild = meta.get('version') != CACHE_VERSION or meta.get('count') != len(json_files)

if need_rebuild:
    ok, fail = 0, 0
    for fp in json_files:
        fname = os.path.basename(fp)
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                data = json.load(f)
            sample = json_to_sample(data)
            if sample is None:
                continue
            sample['source_json'] = fp
            sample['pt_name'] = fname.replace('.json', '.pt')
            torch.save(sample, os.path.join(OUT_DIR, sample['pt_name']))
            ok += 1
        except Exception as exc:
            fail += 1
            print(f'[FAIL] {fname}: {exc}')
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({
            'version': CACHE_VERSION,
            'count': len(json_files),
            'ok': ok,
            'fail': fail,
            'json_root': JSON_ROOT,
        }, f, indent=2, ensure_ascii=False)
    print(f'预处理完成: 成功 {ok}, 失败 {fail}, 输出 {OUT_DIR}')
else:
    print(f'命中缓存 {CACHE_VERSION}，跳过预处理（共 {len(json_files)} 个 JSON）')


In [ ]:

# Step 3: CVAE 模型（GNN 编码 + 条件注入 3D 解码）
LATENT_DIM = 256
HIDDEN_DIM = 128


def build_hetero_convs(in_dim, out_dim, vertical_gat=False):
    h_conv = {}
    node_types = ['living', 'service', 'circulation']
    for s in node_types:
        for d in node_types:
            h_conv[(s, 'horizontal', d)] = SAGEConv(in_dim, out_dim)
    vertical_pairs = [
        ('circulation', 'living'), ('living', 'circulation'),
        ('circulation', 'service'), ('service', 'circulation'),
        ('circulation', 'circulation'),
    ]
    for s, d in vertical_pairs:
        if vertical_gat:
            h_conv[(s, 'vertical', d)] = GATv2Conv(
                in_dim, out_dim, heads=2, concat=False, add_self_loops=False
            )
        else:
            h_conv[(s, 'vertical', d)] = SAGEConv(in_dim, out_dim)
    return HeteroConv(h_conv, aggr='sum')


class HeteroGraphVAEEncoder(nn.Module):
    def __init__(self, node_in_dim=NODE_IN_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM):
        super().__init__()
        self.conv1 = build_hetero_convs(node_in_dim, hidden_dim, vertical_gat=True)
        self.conv2 = build_hetero_convs(hidden_dim, hidden_dim, vertical_gat=False)
        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)

    def _pool(self, x_dict, batch_dict):
        feats = []
        for ntype, x in x_dict.items():
            if ntype in batch_dict and x.size(0) > 0:
                feats.append(global_mean_pool(x, batch_dict[ntype]))
        if not feats:
            dev = x_dict[list(x_dict.keys())[0]].device
            return torch.zeros(1, HIDDEN_DIM, device=dev)
        return torch.stack(feats, dim=0).mean(dim=0)

    def forward(self, x_dict, edge_index_dict, batch_dict):
        x_dict = {k: torch.relu(self.conv1(x_dict, edge_index_dict)[k]) for k in x_dict}
        x_dict = {k: torch.relu(self.conv2(x_dict, edge_index_dict)[k]) for k in x_dict}
        h = self._pool(x_dict, batch_dict)
        return self.to_mu(h), self.to_logvar(h)


class ConditionalVoxelDecoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, cond_dim=COND_DIM, channels=NUM_CHANNELS):
        super().__init__()
        self.init_volume_size = (256, 4, 8, 2)
        in_dim = latent_dim + cond_dim
        self.fc = nn.Linear(in_dim, int(np.prod(self.init_volume_size)))
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(256, 128, 4, stride=2, padding=1), nn.BatchNorm3d(128), nn.ReLU(True),
            nn.ConvTranspose3d(128, 64, 4, stride=2, padding=1), nn.BatchNorm3d(64), nn.ReLU(True),
            nn.ConvTranspose3d(64, 32, 4, stride=2, padding=1), nn.BatchNorm3d(32), nn.ReLU(True),
            nn.ConvTranspose3d(32, channels, 4, stride=2, padding=1),
        )

    def forward(self, z, cond):
        x = torch.cat([z, cond], dim=-1)
        d = self.fc(x).view(z.size(0), *self.init_volume_size)
        return self.decoder(d)


class SpatialModalCVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = HeteroGraphVAEEncoder()
        self.decoder = ConditionalVoxelDecoder()

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, batch):
        mu, logvar = self.encoder(
            batch.x_dict, batch.edge_index_dict, graph_batch_dict(batch)
        )
        z = self.reparameterize(mu, logvar)
        cond = graph_condition(batch)
        logits = self.decoder(z, cond)
        return logits, mu, logvar


model = SpatialModalCVAE().to(device)
print(f'参数量: {sum(p.numel() for p in model.parameters()):,}')


## Step R: 快速恢复（推荐）

**Colab 重启后内存会清空**，Step R **不能作为第一个 cell 单独运行**（否则会 `NameError: os` 或缺少 `SpatialModalCVAE`）。

| 场景 | 需要运行 |
|---|---|
| 首次完整训练 | Step 0 → 1 → 2 → 3 → 4 |
| **Colab 重启 / 断线后** | Step 0（若需）→ **1 → 3 → R** → Step 6/7/8 |
| 同一次 session 内已跑过 1+3 | **Step R** → Step 6a → 6b/8 |
| 改了 Step 1 函数 | Step 1 → **Step R** → 目标 Step |
| 改了模型结构 Step 3 | Step 3 → 4（重训）或 Step 3 → **R** |

> **如何同步代码**：在 Colab 用本地最新版 **`260523-74条件生成与量化评估（MVP）.ipynb` 整文件上传/覆盖**，不要只复制某一个 cell 片段。


In [ ]:
# Step R: 快速恢复环境 + 加载权重（~10 秒）
# Colab 重启后请先跑 Step 1 → Step 1b → Step 3，再执行本 cell

import os, json, glob
from datetime import datetime
import torch

if 'device' not in globals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if 'DATA_DIR' not in globals():
    USE_DRIVE = True
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
        except ImportError:
            USE_DRIVE = False

    def resolve_data_dir():
        candidates = [
            '/content/drive/MyDrive/master_thesis/data',
            '/content/drive/MyDrive/260509_dataset',
            '/content/data',
            os.path.join(os.getcwd(), 'data'),
            os.path.abspath(os.path.join(os.getcwd(), '..', 'data')),
        ]
        for path in candidates:
            if os.path.isdir(path):
                return path
        return candidates[0]

    DATA_DIR = resolve_data_dir()
    JSON_ROOT = os.path.join(DATA_DIR, 'processed')
    OUT_DIR = os.path.join(DATA_DIR, 'processed_tensors_v470')
    WEIGHT_FILENAME = 'spatial_modal_cvae_470_v1.pth'

    def discover_json_files(root=None):
        root = root or JSON_ROOT
        if not os.path.isdir(root):
            return []
        return sorted(
            f for f in glob.glob(os.path.join(root, '**', 'house_*.json'), recursive=True)
            if not f.endswith('_topology.json')
        )

if 'set_experiment_meta' not in globals():
    TOTAL_JSON_COUNT = len(discover_json_files()) if 'discover_json_files' in globals() else 0
    TRAIN_SAMPLE_COUNT = None
    TRAIN_TIMESTAMP = None
    EXPERIMENT_META = {}

    def set_experiment_meta(train_count, total_count=None, timestamp=None, extra=None):
        global TRAIN_SAMPLE_COUNT, TRAIN_TIMESTAMP, EXPERIMENT_META, TOTAL_JSON_COUNT
        TRAIN_SAMPLE_COUNT = int(train_count)
        if total_count is not None:
            TOTAL_JSON_COUNT = int(total_count)
        TRAIN_TIMESTAMP = timestamp or datetime.now().strftime('%Y%m%d_%H%M%S')
        EXPERIMENT_META = {
            'train_samples': TRAIN_SAMPLE_COUNT,
            'total_json': TOTAL_JSON_COUNT,
            'trained_at': TRAIN_TIMESTAMP,
        }
        if extra:
            EXPERIMENT_META.update(extra)
        return EXPERIMENT_META

_restore_deps = {
    'SpatialModalCVAE': 'Step 3（模型定义）',
    'prepare_graph_batch': 'Step 1（工具函数）',
    'forward_model': 'Step 1（工具函数）',
}
_missing = [f'{name} ← 请先运行 {step}' for name, step in _restore_deps.items() if name not in globals()]
if _missing:
    raise RuntimeError(
        'Colab 重启后内存已清空，不能只跑 Step R。\n'
        '请依次执行：Step 0（如需）→ Step 1 → Step 1b → Step 3 → 再回来执行 Step R\n'
        '当前缺少：\n  - ' + '\n  - '.join(_missing)
    )

WEIGHT_PATH = weight_path() if 'weight_path' in globals() else os.path.join(DATA_DIR, WEIGHT_FILENAME)

def quick_restore(force_rebuild_model=True):
    global model
    if force_rebuild_model or 'model' not in globals():
        model = SpatialModalCVAE().to(device)
    if os.path.exists(WEIGHT_PATH):
        model.load_state_dict(torch.load(WEIGHT_PATH, map_location=device, weights_only=True))
        print(f'已加载权重: {WEIGHT_PATH}')
    else:
        print(f'警告: 未找到权重 {WEIGHT_PATH}，请先运行 Step 4 训练')
    model.eval()

    meta_files = sorted(glob.glob(os.path.join(DATA_DIR, 'eval_reports', 'experiment_meta_*.json')))
    if meta_files:
        with open(meta_files[-1], 'r', encoding='utf-8') as f:
            meta = json.load(f)
        set_experiment_meta(meta.get('train_samples', 0), meta.get('total_json'), meta.get('trained_at'), meta)
        print('实验元数据:', EXPERIMENT_META)
    if 'discover_json_files' in globals():
        TOTAL_JSON_COUNT = len(discover_json_files())
    print(f'设备: {device} | 数据: {DATA_DIR} | JSON 数: {TOTAL_JSON_COUNT}')
    return model

quick_restore()


In [ ]:
# Step 4: 训练（AMP + 梯度积累 + KL Annealing + 拓扑加噪 + 合成探针）
import gc
from torch.utils.data import Dataset

_need = ['set_seed', 'model', 'forward_model', 'SPLIT_SEED', 'OUT_DIR', 'augment_batch_topology', 'run_synthetic_probe_eval']
_missing = [k for k in _need if k not in globals()]
if _missing:
    raise RuntimeError(
        '请先运行 Step 1 → Step 1b → Step 3，再执行 Step 4。\n'
        '（内核崩溃/重启后内存会清空，不能直接单跑本 cell）\n'
        '缺少: ' + ', '.join(_missing)
    )

set_seed(SPLIT_SEED)

class LazyPtGraphDataset(Dataset):
    """按 batch 从磁盘懒加载 .pt，避免 468 样本全进 RAM（约 6GB+）。"""
    def __init__(self, entries):
        self.entries = entries  # [(pt_name, filepath), ...]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        pt_name, fp = self.entries[idx]
        sample = torch.load(fp, map_location='cpu', weights_only=False)
        hg = sample['graph']
        hg.y = sample['voxel']
        cond = sample['condition']
        hg.condition = cond.unsqueeze(0) if cond.dim() == 1 else cond
        hg.pt_name = sample.get('pt_name') or pt_name
        return hg


pt_files = sorted([
    p for p in glob.glob(os.path.join(OUT_DIR, '*.pt'))
    if not os.path.basename(p).startswith('_')
])
if not pt_files:
    raise RuntimeError('请先运行 Step 2 预处理')

items = [(os.path.basename(fp), fp) for fp in pt_files]
pt_names = [x[0] for x in items]
train_names, val_names, split_meta = load_train_val_split(pt_names)
train_name_set, val_name_set = set(train_names), set(val_names)
train_entries = [x for x in items if x[0] in train_name_set]
val_entries = [x for x in items if x[0] in val_name_set]
print(f'样本总数 {len(items)} | 训练 {len(train_entries)} | 验证 {len(val_entries)}')
print(f'划分文件: {split_path()}')
print('数据加载: 懒加载（按 batch 读盘，节省 RAM）')

train_loader = GraphDataLoader(LazyPtGraphDataset(train_entries), batch_size=BATCH_SIZE, shuffle=True)
val_loader = GraphDataLoader(LazyPtGraphDataset(val_entries), batch_size=BATCH_SIZE, shuffle=False) if val_entries else None

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)
_use_amp = USE_AMP and device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=_use_amp)

import time as _time
save_path = weight_path()
ckpt_path = checkpoint_path()

best_val = float('inf')
bad_epochs = 0
history = {'train': [], 'val': [], 'kl_beta': [], 'synth_probe': []}
_train_start = _time.time()
start_epoch = 1


def save_training_checkpoint(epoch, tag='latest'):
    payload = {
        'epoch': epoch,
        'cache_version': CACHE_VERSION,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_val': best_val,
        'bad_epochs': bad_epochs,
        'history': history,
        'train_start': _train_start,
        'train_count': len(train_entries),
        'val_count': len(val_entries),
        'tag': tag,
    }
    tmp_path = ckpt_path + '.tmp'
    torch.save(payload, tmp_path)
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)
    os.rename(tmp_path, ckpt_path)


def try_resume_training():
    global start_epoch, best_val, bad_epochs, history, _train_start
    if FORCE_NEW_TRAINING:
        print('FORCE_NEW_TRAINING=True，从头训练')
        return
    if not RESUME_IF_CHECKPOINT:
        print('RESUME_IF_CHECKPOINT=False，从头训练')
        return
    if not os.path.exists(ckpt_path):
        if MANUAL_RESUME_EPOCH > 0 and os.path.exists(save_path):
            model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
            start_epoch = int(MANUAL_RESUME_EPOCH) + 1
            print(f'⚠ 无检查点，从最佳权重续训: 已完成 epoch {MANUAL_RESUME_EPOCH} → 从 {start_epoch} 继续')
            print('  （优化器状态未恢复，loss 可能短暂抖动；之后每 epoch 会自动存检查点）')
            return
        print('无检查点，从头训练')
        return
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    if ckpt.get('cache_version') != CACHE_VERSION:
        print(f"检查点版本 {ckpt.get('cache_version')} != {CACHE_VERSION}，从头训练")
        return
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = int(ckpt['epoch']) + 1
    best_val = float(ckpt['best_val'])
    bad_epochs = int(ckpt['bad_epochs'])
    history = ckpt.get('history', history)
    _train_start = ckpt.get('train_start', _time.time())
    torch.save(model.state_dict(), save_path)
    print(f"✓ 断点续训: 已完成 epoch {ckpt['epoch']} → 从 {start_epoch}/{EPOCHS} 继续")
    print(f"  best_val={best_val:.4f} | bad_epochs={bad_epochs} | 检查点: {ckpt_path}")


try_resume_training()


def kl_beta_for_epoch(epoch):
    if KL_ANNEAL_EPOCHS <= 0:
        return KL_WEIGHT_MAX
    t = min(1.0, epoch / KL_ANNEAL_EPOCHS)
    return KL_WEIGHT_MAX * t


def compute_batch_loss(batch, kl_weight):
    bs = graph_batch_size(batch)
    target = batch.y.view(bs, NUM_CHANNELS, RES_X, RES_Y, RES_Z)
    logits, mu, logvar = forward_model(model, batch)
    bce = F.binary_cross_entropy_with_logits(logits, target)
    dice = dice_loss_with_logits(logits, target)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    loss = bce + DICE_WEIGHT * dice + kl_weight * kl
    return loss, bs


def run_epoch(loader, train_mode=True, epoch=1):
    model.train(train_mode)
    total = 0.0
    n = 0
    kl_weight = kl_beta_for_epoch(epoch) if train_mode else KL_WEIGHT_MAX
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, start=1):
        batch = batch.to(device)
        if train_mode and AUGMENT_PROB > 0 and random.random() < AUGMENT_PROB:
            batch = augment_batch_topology(batch)

        with torch.amp.autocast('cuda', enabled=_use_amp):
            loss, bs = compute_batch_loss(batch, kl_weight)
            loss_scaled = loss / ACCUM_STEPS

        if train_mode:
            scaler.scale(loss_scaled).backward()
            if step % ACCUM_STEPS == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        total += loss.item() * bs
        n += bs
        del batch, loss, loss_scaled
    if train_mode and device.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return total / max(n, 1), kl_weight


if start_epoch > EPOCHS:
    print(f'检查点显示已完成 {EPOCHS} epoch，跳过训练。可直接 Step R / Step 7。')
    epoch = EPOCHS
else:
    print(f'开始训练 ({device}) | AMP={USE_AMP and device.type == "cuda"} | accum={ACCUM_STEPS} | eff_batch≈{BATCH_SIZE * ACCUM_STEPS}')
    print(f'epoch 范围: {start_epoch}–{EPOCHS} | 检查点: {ckpt_path}')
    for epoch in range(start_epoch, EPOCHS + 1):
        tr, beta = run_epoch(train_loader, True, epoch)
        history['train'].append(tr)
        history['kl_beta'].append(beta)
        va, _ = run_epoch(val_loader, False, epoch) if val_loader else (tr, beta)
        history['val'].append(va)
        scheduler.step(va)

        if va < best_val:
            best_val = va
            bad_epochs = 0
            torch.save(model.state_dict(), save_path)
        else:
            bad_epochs += 1

        save_training_checkpoint(epoch)

        show_log = (epoch == start_epoch or epoch % LOG_EVERY == 0 or epoch % 10 == 0)
        if show_log:
            print(f'Epoch {epoch:03d}/{EPOCHS} | train={tr:.4f} val={va:.4f} best={best_val:.4f} beta={beta:.2e} | ckpt✓')
        if epoch % PROBE_EVERY == 0:
            print('  [合成拓扑探针]')
            probe = run_synthetic_probe_eval(model, verbose=True)
            history['synth_probe'].append({'epoch': epoch, **probe})

        if bad_epochs >= PATIENCE:
            print(f'早停于 epoch {epoch}，最佳 val={best_val:.4f}')
            break

    done_path = ckpt_path.replace('.pt', '_done.pt')
    if os.path.exists(ckpt_path):
        os.rename(ckpt_path, done_path)
        print(f'训练结束，检查点归档: {done_path}')

set_experiment_meta(
    train_count=len(train_entries),
    total_count=len(items),
    extra={
        'val_samples': len(val_entries),
        'epochs_run': epoch,
        'best_val_loss': float(best_val),
        'train_seconds': round(_time.time() - _train_start, 1),
        'cache_version': CACHE_VERSION,
        'weight_file': WEIGHT_FILENAME,
        'split_file': SPLIT_FILENAME,
        'batch_size': BATCH_SIZE,
        'accum_steps': ACCUM_STEPS,
        'use_amp': USE_AMP,
        'kl_weight_max': KL_WEIGHT_MAX,
        'kl_anneal_epochs': KL_ANNEAL_EPOCHS,
        'edge_drop_prob': EDGE_DROP_PROB,
        'augment_prob': AUGMENT_PROB,
        'synth_probe': history['synth_probe'],
        'train_names': train_names,
        'val_names': val_names,
    },
)
meta_path = experiment_report_path('experiment_meta', ext='json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(EXPERIMENT_META, f, indent=2, ensure_ascii=False)
print(f'最佳权重已保存: {save_path}')
print(f'实验元数据: {meta_path}')
print(EXPERIMENT_META)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train'], label='train')
axes[0].plot(history['val'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].plot(history['kl_beta'], label='kl_beta', color='tab:orange')
axes[1].set_title('KL beta schedule')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Step 5: 重建基线 — GT 异构图 → 体素（性能上界）
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'colab'

COLOR_DICT = {
    1: '#808080', 2: '#FF8000', 3: '#FFFF00', 4: '#00FF00', 5: '#0000FF',
    6: '#FF0000', 7: '#B0B0FF', 8: '#A000FF', 9: '#3CB371', 10: '#00FFFF', 11: '#FFC0CB',
}
NAME_MAP = {
    1: '玄关', 2: '客厅', 3: '餐厅', 4: '厨房', 5: '卧室', 6: '卫生间',
    7: '过道', 8: '楼梯', 9: '储藏', 10: '阳台', 11: '多功能',
}

weight_path = weight_path() if 'weight_path' in globals() else os.path.join(DATA_DIR, WEIGHT_FILENAME)
model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
model.eval()

sample_pt = random.choice(pt_files)
sample = torch.load(sample_pt, weights_only=False)
hg = sample['graph']
hg.y = sample['voxel']
hg.condition = sample['condition'].unsqueeze(0)
batch = prepare_graph_batch(hg, condition=sample['condition']).to(device)

with torch.no_grad():
    logits, _, _ = forward_model(model, batch, sample['condition'])
    pred = torch.argmax(logits[0], dim=0).cpu().numpy()
    gt = torch.argmax(sample['voxel'], dim=0).numpy()

def voxel_coords(class_map, max_pts=8000):
    coords = np.argwhere(class_map > 0)
    if len(coords) > max_pts:
        coords = coords[np.random.choice(len(coords), max_pts, replace=False)]
    xs, ys, zs, cols, texts = [], [], [], [], []
    for ix, iy, iz in coords:
        cid = int(class_map[ix, iy, iz])
        xs.append(ix * VOXEL_SIZE)
        ys.append(iy * VOXEL_SIZE)
        zs.append(iz * VOXEL_SIZE)
        cols.append(COLOR_DICT.get(cid, '#CCC'))
        texts.append(NAME_MAP.get(cid, str(cid)))
    return xs, ys, zs, cols, texts

from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scene'}, {'type': 'scene'}]],
                    subplot_titles=('Ground Truth', 'CVAE Prediction'))

for col, arr in ((1, gt), (2, pred)):
    xs, ys, zs, cols, texts = voxel_coords(arr)
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode='markers',
        marker=dict(size=3, color=cols, symbol='square', opacity=0.85),
        text=texts, hoverinfo='text', showlegend=False,
    ), row=1, col=col)

acc = (pred == gt).mean()
occupied_acc = (pred[gt > 0] == gt[gt > 0]).mean() if (gt > 0).any() else 0.0
print(f'样本: {os.path.basename(sample_pt)}')
print(f'全网格准确率: {acc*100:.2f}% | 非空体素准确率: {occupied_acc*100:.2f}%')

fig.update_layout(height=650, width=1200, title='空间模态 CVAE 体素重建对比')
fig.update_scenes(aspectmode='data')
fig.show()


## Step 6: 条件生成管线

从 **用地尺寸 + 功能清单** 构建合成异构图与条件向量，调用训练好的 CVAE 生成体素，再提取并规整为 300mm 模数体块。


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.colors as mcolors
# Step 6a: 条件生成核心函数
import math
import networkx as nx

DEFAULT_ROOM_SIZE = {
    'entryway': (2400, 2400, 3000),
    'living_room': (6000, 4500, 3000),
    'dining_room': (3600, 3300, 3000),
    'kitchen': (3300, 3000, 3000),
    'bedroom': (3600, 3600, 3000),
    'bathroom': (2400, 2400, 3000),
    'corridor': (1800, 2400, 3000),
    'stairs': (3000, 3000, 6000),
    'utility': (2400, 2400, 3000),
    'balcony': (3000, 1800, 3000),
    'multi_purpose': (3600, 3300, 3000),
}

TYPE_COLOR_DICT = {
    'entryway': '#808080', 'living_room': '#FF8000', 'dining_room': '#FFFF00',
    'kitchen': '#00FF00', 'bedroom': '#0000FF', 'bathroom': '#FF0000',
    'corridor': '#B0B0FF', 'stairs': '#A000FF', 'utility': '#3CB371',
    'balcony': '#00FFFF', 'multi_purpose': '#FFC0CB',
}

CN_NAMES = {
    'entryway': '玄关', 'living_room': '客厅', 'dining_room': '餐厅', 'kitchen': '厨房',
    'bedroom': '卧室', 'bathroom': '卫生间', 'corridor': '过道', 'stairs': '楼梯',
    'utility': '储藏', 'balcony': '阳台', 'multi_purpose': '多功能',
}


def snap_modulus(v):
    return round(float(v) / VOXEL_SIZE) * VOXEL_SIZE


def build_user_request(site_x, site_y, room_counts, site_z=6000):
    counts = {k: int(v) for k, v in room_counts.items() if int(v) > 0}
    return {
        'site_x': float(site_x), 'site_y': float(site_y), 'site_z': float(site_z),
        'room_counts': counts,
    }


def build_program_topology(room_counts, seed=42):
    G = nx.Graph()
    nodes = []
    bath_i = corr_i = 0
    for r_type, count in room_counts.items():
        for i in range(count):
            nid = f"{r_type}_{i}"
            if r_type in ['entryway', 'living_room', 'dining_room', 'kitchen']:
                floor = 1
            elif r_type in ['bedroom', 'balcony']:
                floor = 2
            elif r_type == 'bathroom':
                floor = 1 if bath_i % 2 == 0 else 2
                bath_i += 1
            elif r_type == 'corridor':
                floor = 1 if corr_i % 2 == 0 else 2
                corr_i += 1
            elif r_type == 'stairs':
                floor = '1&2'
            else:
                floor = 1
            nodes.append((nid, r_type, floor))
            G.add_node(nid, type=r_type, floor=floor)

    f1_corr = [n for n, t, f in nodes if t == 'corridor' and f == 1]
    f2_corr = [n for n, t, f in nodes if t == 'corridor' and f == 2]
    f1_hub = f1_corr[0] if f1_corr else next((n for n, t, f in nodes if t == 'living_room'), nodes[0][0])
    f2_hub = f2_corr[0] if f2_corr else next((n for n, t, f in nodes if t == 'bedroom'), nodes[-1][0])

    edge_types = {}
    for nid, r_type, floor in nodes:
        if r_type == 'stairs':
            continue
        hub = f1_hub if floor == 1 else f2_hub
        if nid != hub:
            G.add_edge(nid, hub)
            edge_types[(nid, hub)] = 'horizontal'
            edge_types[(hub, nid)] = 'horizontal'

    stairs = [n for n, t, f in nodes if t == 'stairs']
    if stairs:
        st = stairs[0]
        if f1_hub != st:
            G.add_edge(st, f1_hub)
            edge_types[(st, f1_hub)] = 'vertical'
        if f2_hub != st and f2_hub != f1_hub:
            G.add_edge(st, f2_hub)
            edge_types[(st, f2_hub)] = 'vertical'

    pos = nx.spring_layout(G, seed=seed, k=1.2)
    return G, pos, nodes, edge_types


def layout_rooms_from_program(user_req, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    G, pos, nodes, edge_types = build_program_topology(user_req['room_counts'], seed=seed)
    sx, sy = user_req['site_x'], user_req['site_y']
    rooms = []
    for nid, r_type, floor in nodes:
        w, d, h = DEFAULT_ROOM_SIZE.get(r_type, (3600, 3600, 3000))
        px, py = pos[nid]
        cx = (px + 1) / 2 * (sx * 0.7) + sx * 0.15
        cy = (py + 1) / 2 * (sy * 0.7) + sy * 0.15
        cx, cy = snap_modulus(cx), snap_modulus(cy)
        w, d = snap_modulus(w), snap_modulus(d)
        z0 = 0 if floor == 1 or floor == '1&2' else 3000
        z1 = z0 + h
        rooms.append({
            'id': nid, 'type': r_type, 'floor': 1 if floor == '1&2' else floor,
            'box_min': [max(0, cx - w / 2), max(0, cy - d / 2), z0],
            'box_max': [min(sx, cx + w / 2), min(sy, cy + d / 2), z1],
            'lighting_access': 'direct' if r_type in ['living_room', 'bedroom', 'balcony'] else 'indirect',
            'lighting_priority': 8 if r_type in ['living_room', 'bedroom'] else 4,
            'effective_lighting': [],
        })
    return rooms, G, pos, edge_types


def request_to_house_json(user_req, rooms):
    stats = {t: 0 for t in ROOM_TYPES}
    shown_types = set()
    for r in rooms:
        stats[r['type']] = stats.get(r['type'], 0) + 1
    return {
        'metadata': {
            'stats': stats,
            'total_rooms': len(rooms),
            'building_size': {'x': user_req['site_x'], 'y': user_req['site_y'], 'z': user_req['site_z']},
            'constraints': {'modulus': int(VOXEL_SIZE)},
        },
        'rooms': rooms,
    }


@torch.no_grad()
def generate_voxels_from_request(user_req, model, seed=42):
    rooms, G, pos, edge_types = layout_rooms_from_program(user_req, seed=seed)
    data = request_to_house_json(user_req, rooms)
    sample = json_to_sample(data)
    hg = sample['graph']
    batch = prepare_graph_batch(hg, condition=sample['condition']).to(device)
    model.eval()
    logits, _, _ = forward_model(model, batch, sample['condition'])
    pred_cls = torch.argmax(logits[0], dim=0).cpu().numpy()
    return pred_cls, sample, rooms, G, pos


def count_components_3d(mask):
    if not mask.any():
        return 0
    visited = np.zeros(mask.shape, dtype=bool)
    n_comp = 0
    xs, ys, zs = np.where(mask)
    for x, y, z in zip(xs, ys, zs):
        if visited[x, y, z]:
            continue
        n_comp += 1
        stack = [(int(x), int(y), int(z))]
        visited[x, y, z] = True
        while stack:
            cx, cy, cz = stack.pop()
            for dx, dy, dz in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)):
                nx, ny, nz = cx+dx, cy+dy, cz+dz
                if 0 <= nx < mask.shape[0] and 0 <= ny < mask.shape[1] and 0 <= nz < mask.shape[2]:
                    if mask[nx, ny, nz] and not visited[nx, ny, nz]:
                        visited[nx, ny, nz] = True
                        stack.append((nx, ny, nz))
    return n_comp


def voxel_grid_phys_offset(rooms):
    """与 json_to_sample 编码一致的 XY/Z 偏移，用于体素→物理坐标反变换。"""
    if not rooms:
        return np.array([0.0, 0.0]), 0.0
    all_coords = np.array([r['box_min'] for r in rooms] + [r['box_max'] for r in rooms])
    build_min = all_coords.min(axis=0)
    build_max = all_coords.max(axis=0)
    phys_center_xy = (build_min[:2] + build_max[:2]) / 2.0
    offset_xy = np.array([RES_X * VOXEL_SIZE / 2, RES_Y * VOXEL_SIZE / 2]) - phys_center_xy
    return offset_xy, float(build_min[2])


def voxels_to_boxes(pred_cls, user_req=None, rooms=None):
    TYPE_NAMES_LOCAL = {v: k for k, v in CHANNEL_MAP.items() if v > 0}
    offset_xy, z_min_phys = voxel_grid_phys_offset(rooms) if rooms else (np.array([0.0, 0.0]), 0.0)
    boxes = []
    for cid in range(1, NUM_CHANNELS):
        mask = pred_cls == cid
        if not mask.any():
            continue
        visited = np.zeros(mask.shape, dtype=bool)
        xs, ys, zs = np.where(mask)
        for x, y, z in zip(xs, ys, zs):
            if visited[x, y, z]:
                continue
            stack = [(int(x), int(y), int(z))]
            comp = []
            visited[x, y, z] = True
            while stack:
                cx, cy, cz = stack.pop()
                comp.append((cx, cy, cz))
                for dx, dy, dz in ((1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)):
                    nx, ny, nz = cx+dx, cy+dy, cz+dz
                    if 0 <= nx < mask.shape[0] and 0 <= ny < mask.shape[1] and 0 <= nz < mask.shape[2]:
                        if mask[nx, ny, nz] and not visited[nx, ny, nz]:
                            visited[nx, ny, nz] = True
                            stack.append((nx, ny, nz))
            arr = np.array(comp)
            ix0, iy0, iz0 = arr.min(axis=0)
            ix1, iy1, iz1 = arr.max(axis=0) + 1
            rtype = TYPE_NAMES_LOCAL[cid]
            boxes.append({
                'type': rtype,
                'box_min': [
                    snap_modulus(ix0 * VOXEL_SIZE - offset_xy[0]),
                    snap_modulus(iy0 * VOXEL_SIZE - offset_xy[1]),
                    snap_modulus(iz0 * VOXEL_SIZE + z_min_phys),
                ],
                'box_max': [
                    snap_modulus(ix1 * VOXEL_SIZE - offset_xy[0]),
                    snap_modulus(iy1 * VOXEL_SIZE - offset_xy[1]),
                    snap_modulus(iz1 * VOXEL_SIZE + z_min_phys),
                ],
            })
    return boxes






def rooms_to_boxes(rooms):
    return [{'type': r['type'], 'box_min': list(r['box_min']), 'box_max': list(r['box_max'])} for r in rooms]


def _setup_matplotlib_cjk():
    """配置 matplotlib 中文字体，避免图例/标题乱码。"""
    import os
    from matplotlib import font_manager
    try:
        names = {f.name for f in font_manager.fontManager.ttflist}
        if not any('Noto Sans CJK' in n for n in names) and os.path.isdir('/usr/share/fonts'):
            import subprocess
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=False, capture_output=True,
            )
            try:
                font_manager._load_fontmanager(try_read_cache=False)
            except Exception:
                font_manager.fontManager = font_manager.FontManager()
    except Exception:
        pass
    for name in (
        'Noto Sans CJK SC', 'Noto Sans CJK JP', 'SimHei', 'Microsoft YaHei',
        'WenQuanYi Micro Hei', 'Arial Unicode MS',
    ):
        if name in {f.name for f in font_manager.fontManager.ttflist}:
            plt.rcParams['font.sans-serif'] = [name, 'DejaVu Sans']
            break
    plt.rcParams['axes.unicode_minus'] = False


def _hex_to_rgba(hex_color, alpha=0.42):
    return mcolors.to_rgba(hex_color, alpha=alpha)


def _box_poly_faces(x0, y0, z0, x1, y1, z1):
    verts = [
        (x0, y0, z0), (x1, y0, z0), (x1, y1, z0), (x0, y1, z0),
        (x0, y0, z1), (x1, y0, z1), (x1, y1, z1), (x0, y1, z1),
    ]
    faces_idx = [
        [0, 1, 2, 3], [4, 5, 6, 7], [0, 1, 5, 4],
        [2, 3, 7, 6], [1, 2, 6, 5], [0, 3, 7, 4],
    ]
    return [[verts[i] for i in idx] for idx in faces_idx]


def _room_centroid(room):
    bm, bx = room['box_min'], room['box_max']
    return ((bm[0] + bx[0]) / 2, (bm[1] + bx[1]) / 2, (bm[2] + bx[2]) / 2)


def _layout_graph_centroids(layout_rooms, rooms):
    topo = layout_rooms if layout_rooms is not None else rooms
    return {r['id']: _room_centroid(r) for r in topo if 'id' in r}


def _draw_boxes_3d(ax, rooms, cmap):
    from matplotlib.lines import Line2D
    legend_handles = {}
    for r in rooms:
        x0, y0, z0 = r['box_min']
        x1, y1, z1 = r['box_max']
        if x1 <= x0 or y1 <= y0 or z1 <= z0:
            continue
        color = cmap.get(r['type'], '#CCCCCC')
        poly = Poly3DCollection(
            _box_poly_faces(x0, y0, z0, x1, y1, z1),
            facecolors=[_hex_to_rgba(color)],
            edgecolors='black',
            linewidths=0.8,
        )
        ax.add_collection3d(poly)
        if r['type'] not in legend_handles:
            legend_handles[r['type']] = Line2D(
                [0], [0], color=color, lw=6,
                label=CN_NAMES.get(r['type'], r['type']),
            )
    return legend_handles


def _draw_graph_3d(ax, graph, centroids, edge_types, cmap):
    from matplotlib.lines import Line2D
    extra_handles = []
    if graph is None or not centroids:
        return extra_handles

    xs, ys, zs, node_colors = [], [], [], []
    for nid in graph.nodes:
        if nid not in centroids:
            continue
        x, y, z = centroids[nid]
        xs.append(x)
        ys.append(y)
        zs.append(z)
        ntype = graph.nodes[nid].get('type', 'corridor')
        node_colors.append(cmap.get(ntype, '#333333'))
    if xs:
        ax.scatter(
            xs, ys, zs, c=node_colors, s=58, depthshade=False,
            edgecolors='white', linewidths=0.9, zorder=10,
        )

    seen = set()
    for u, v in graph.edges:
        if u not in centroids or v not in centroids:
            continue
        p0, p1 = centroids[u], centroids[v]
        et = (edge_types or {}).get((u, v), 'horizontal')
        if et == 'vertical':
            color, ls, lw, tag = '#E94F37', '--', 2.0, '垂直连接'
        else:
            color, ls, lw, tag = '#B03060', '--', 1.6, '水平连接'
        ax.plot(
            [p0[0], p1[0]], [p0[1], p1[1]], [p0[2], p1[2]],
            color=color, linestyle=ls, linewidth=lw, zorder=8,
        )
        if tag not in seen:
            extra_handles.append(Line2D([0], [0], color=color, lw=lw, linestyle=ls, label=tag))
            seen.add(tag)
    return extra_handles


def _set_equal_aspect_3d(ax, x0, x1, y0, y1, z0, z1):
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_zlim(z0, z1)
    ax.set_box_aspect((max(x1 - x0, 1.0), max(y1 - y0, 1.0), max(z1 - z0, 1.0)))


def plot_3d_layout(
    rooms,
    site_x=None,
    site_y=None,
    title='AI 生成的 3D 功能体块布局图',
    graph=None,
    edge_types=None,
    layout_rooms=None,
    color_map=None,
    site_z=9000,
):
    """matplotlib 静态 3D：半透明体块 + 质心节点 + 拓扑边（参考论文配图风格）。"""
    _setup_matplotlib_cjk()
    cmap = color_map or TYPE_COLOR_DICT
    fig = plt.figure(figsize=(10, 7.5))
    ax = fig.add_subplot(111, projection='3d')

    legend_handles = _draw_boxes_3d(ax, rooms, cmap)
    centroids = _layout_graph_centroids(layout_rooms, rooms)
    edge_handles = _draw_graph_3d(ax, graph, centroids, edge_types, cmap)

    if site_x and site_y:
        _set_equal_aspect_3d(ax, 0, float(site_x), 0, float(site_y), 0, float(site_z))
    elif rooms:
        pts = np.array([r['box_min'] + r['box_max'] for r in rooms if r['box_max'][0] > r['box_min'][0]])
        if len(pts):
            mins, maxs = pts.min(axis=0), pts.max(axis=0)
            pad = max(max(maxs - mins) * 0.05, 300.0)
            _set_equal_aspect_3d(
                ax,
                mins[0] - pad, maxs[0] + pad,
                mins[1] - pad, maxs[1] + pad,
                mins[2] - pad, maxs[2] + pad,
            )

    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Y (mm)')
    ax.set_zlabel('Z (mm)')
    ax.set_title(title)
    ax.view_init(elev=22, azim=-58)
    ax.grid(True, linestyle='-', alpha=0.22)
    ax.legend(
        handles=list(legend_handles.values()) + edge_handles,
        loc='upper left', fontsize=9, framealpha=0.92,
    )
    fig.tight_layout()
    return fig


def plot_2d_floorplan(
    rooms,
    site_x=None,
    site_y=None,
    title='平面功能布局图',
    graph=None,
    edge_types=None,
    layout_rooms=None,
    color_map=None,
):
    """静态平面图（XY 俯视）：半透明矩形 + 质心节点 + 拓扑边。"""
    from matplotlib.patches import Rectangle
    from matplotlib.lines import Line2D

    _setup_matplotlib_cjk()
    cmap = color_map or TYPE_COLOR_DICT
    fig, ax = plt.subplots(figsize=(9, 7.2))

    legend_handles = {}
    for r in rooms:
        x0, y0 = r['box_min'][0], r['box_min'][1]
        x1, y1 = r['box_max'][0], r['box_max'][1]
        if x1 <= x0 or y1 <= y0:
            continue
        color = cmap.get(r['type'], '#CCCCCC')
        ax.add_patch(Rectangle(
            (x0, y0), x1 - x0, y1 - y0,
            facecolor=_hex_to_rgba(color), edgecolor='black', linewidth=0.9, zorder=1,
        ))
        if r['type'] not in legend_handles:
            legend_handles[r['type']] = Line2D(
                [0], [0], color=color, lw=6,
                label=CN_NAMES.get(r['type'], r['type']),
            )

    centroids = _layout_graph_centroids(layout_rooms, rooms)
    edge_handles = []
    if graph is not None and centroids:
        xs, ys, node_colors = [], [], []
        for nid in graph.nodes:
            if nid not in centroids:
                continue
            cx, cy, _ = centroids[nid]
            xs.append(cx)
            ys.append(cy)
            ntype = graph.nodes[nid].get('type', 'corridor')
            node_colors.append(cmap.get(ntype, '#333333'))
        if xs:
            ax.scatter(xs, ys, c=node_colors, s=50, edgecolors='white', linewidths=0.8, zorder=5)

        seen = set()
        for u, v in graph.edges:
            if u not in centroids or v not in centroids:
                continue
            p0, p1 = centroids[u], centroids[v]
            et = (edge_types or {}).get((u, v), 'horizontal')
            if et == 'vertical':
                color, ls, lw, tag = '#E94F37', '--', 1.8, '垂直连接'
            else:
                color, ls, lw, tag = '#B03060', '--', 1.4, '水平连接'
            ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=color, linestyle=ls, linewidth=lw, zorder=4)
            if tag not in seen:
                edge_handles.append(Line2D([0], [0], color=color, lw=lw, linestyle=ls, label=tag))
                seen.add(tag)

    if site_x and site_y:
        ax.set_xlim(0, site_x)
        ax.set_ylim(0, site_y)
    else:
        ax.autoscale()
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Y (mm)')
    ax.set_title(title)
    ax.grid(True, linestyle='-', alpha=0.28)
    ax.legend(
        handles=list(legend_handles.values()) + edge_handles,
        loc='upper right', fontsize=9, framealpha=0.92,
    )
    fig.tight_layout()
    return fig


def show_layout_fig(fig):
    """显示 matplotlib 图；可传入单图或 (3D, 2D) 元组。"""
    from IPython.display import display
    if isinstance(fig, (list, tuple)):
        for f in fig:
            display(f)
    else:
        display(fig)


@torch.no_grad()
def infer_voxels_multi_strategy(user_req, rooms, model):
    """尝试多种解码策略，缓解合成拓扑下体素全空的问题"""
    data = request_to_house_json(user_req, rooms)
    sample = json_to_sample(data)
    hg = sample['graph']
    cond = sample['condition']
    batch = prepare_graph_batch(hg, condition=cond).to(device)
    cond_dev = cond.unsqueeze(0).to(device) if cond.dim() == 1 else cond.to(device)
    model.eval()

    candidates = []
    logits, mu, logvar = forward_model(model, batch, cond)
    pred_graph = torch.argmax(logits[0], dim=0).cpu().numpy()
    candidates.append(('graph_forward', pred_graph, int((pred_graph > 0).sum())))

    logits_mu = model.decoder(mu, cond_dev)
    pred_mu = torch.argmax(logits_mu[0], dim=0).cpu().numpy()
    candidates.append(('encoder_mu', pred_mu, int((pred_mu > 0).sum())))

    z_rand = torch.randn(1, LATENT_DIM, device=device)
    logits_rand = model.decoder(z_rand, cond_dev)
    pred_rand = torch.argmax(logits_rand[0], dim=0).cpu().numpy()
    candidates.append(('random_z_cond', pred_rand, int((pred_rand > 0).sum())))

    best = max(candidates, key=lambda x: x[2])
    return best[1], best[2], best[0], candidates


def generate_user_layout(user_req, model, seed=None):
    if seed is None:
        seed = random.randint(1, 999999)
    rooms, G, pos, edge_types = layout_rooms_from_program(user_req, seed=seed)
    pred, n_occ, mode, _ = infer_voxels_multi_strategy(user_req, rooms, model)
    if n_occ > 0:
        display_rooms = voxels_to_boxes(pred, user_req, rooms)
        display_source = 'model_voxels'
    else:
        display_rooms = rooms
        display_source = 'layout_fallback'
    return {
        'rooms': rooms, 'display_rooms': display_rooms, 'display_source': display_source,
        'graph': G, 'pos': pos, 'edge_types': edge_types, 'pred': pred,
        'n_occ': n_occ, 'decode_mode': mode, 'seed': seed,
    }


def count_rooms_by_type(boxes):
    counts = {t: 0 for t in ROOM_TYPES}
    for b in boxes:
        counts[b['type']] = counts.get(b['type'], 0) + 1
    return counts


def program_count_mae(requested, predicted):
    keys = set(requested.keys()) | set(predicted.keys())
    diffs = [abs(int(requested.get(k, 0)) - int(predicted.get(k, 0))) for k in keys]
    return float(np.mean(diffs)) if diffs else 0.0


def program_count_acc(requested, predicted):
    keys = set(requested.keys()) | set(predicted.keys())
    ok, total = 0, 0
    for k in keys:
        total += max(int(requested.get(k, 0)), 1)
        ok += min(int(requested.get(k, 0)), int(predicted.get(k, 0)))
    return float(ok / max(total, 1))


def modulus_compliance(boxes):
    if not boxes:
        return 0.0
    ok = total = 0
    for b in boxes:
        for k in ('box_min', 'box_max'):
            for v in b[k]:
                total += 1
                if abs(v / VOXEL_SIZE - round(v / VOXEL_SIZE)) < 1e-6:
                    ok += 1
    return ok / max(total, 1)


def site_fit_rate(boxes, site_x, site_y, site_z=9000):
    if not boxes:
        return 0.0
    ok = 0
    for b in boxes:
        bm, bx = b['box_min'], b['box_max']
        if bm[0] >= -1 and bm[1] >= -1 and bx[0] <= site_x + 1 and bx[1] <= site_y + 1 and bx[2] <= site_z + 1:
            ok += 1
    return ok / len(boxes)


def ensure_model_loaded(force_rebuild_model=False):
    """加载训练权重；未跑 Step R 时也可在 6b/8 前调用。"""
    global model
    _deps = {
        'SpatialModalCVAE': 'Step 3',
        'DATA_DIR': 'Step 1',
        'WEIGHT_FILENAME': 'Step 1',
        'device': 'Step 1',
    }
    _missing = [f'{k} ← {v}' for k, v in _deps.items() if k not in globals()]
    if _missing:
        raise RuntimeError(
            'Colab 重启后内存会清空，不能只跑 Step 6b/8。\n'
            '请依次执行：Step 1 → Step 1b → Step 3 → Step 6a → 再跑 6b/8\n'
            '（或先跑 Step R 加载权重与元数据）\n'
            '当前缺少：\n  - ' + '\n  - '.join(_missing)
        )
    import os
    wp = weight_path() if 'weight_path' in globals() else os.path.join(DATA_DIR, WEIGHT_FILENAME)
    if force_rebuild_model or 'model' not in globals():
        model = SpatialModalCVAE().to(device)
    if os.path.exists(wp):
        model.load_state_dict(torch.load(wp, map_location=device, weights_only=True))
        print(f'已加载权重: {wp}')
    else:
        print(f'警告: 未找到权重 {wp}，请先运行 Step 4 训练')
    model.eval()
    return model


print('Step 6a 就绪: plot_3d_layout / generate_user_layout / ensure_model_loaded')





In [ ]:
# Step 6b: 单次条件生成 + matplotlib 3D 预览
ensure_model_loaded(force_rebuild_model=False)

demo_req = build_user_request(
    site_x=18000, site_y=15000,
    room_counts={
        'entryway': 1, 'living_room': 1, 'dining_room': 1, 'kitchen': 1,
        'bedroom': 3, 'bathroom': 2, 'corridor': 2, 'stairs': 1, 'balcony': 1,
    },
)
result = generate_user_layout(demo_req, model, seed=123)
print('seed:', result['seed'])
print('解码策略:', result['decode_mode'], '| 非空体素:', result['n_occ'])
plot_rooms = result.get('display_rooms', result['rooms'])
print('显示来源:', result.get('display_source', 'rooms'), '| 体块数:', len(plot_rooms))
_plot_kw = dict(
    graph=result.get('graph'), edge_types=result.get('edge_types'), layout_rooms=result.get('rooms'),
)
fig3d = plot_3d_layout(plot_rooms, 18000, 15000, title='条件生成三维功能体块', **_plot_kw)
fig2d = plot_2d_floorplan(plot_rooms, 18000, 15000, title='平面功能布局图', **_plot_kw)
show_layout_fig((fig3d, fig2d))


## Step 7: 量化评估（重建 vs 条件生成 best-of-K）

- **重建模式**：GT 异构图（上界）
- **生成模式**：仅用户约束 → 合成拓扑，**K 次采样取最优**（真实使用路径）
- 验证集与训练共用 `train_val_split_v470.json`

导出命名：`{报告名}_json{总数}_train{训练数}_{时间}.csv`


In [ ]:
# Step 7a: 量化指标函数
EVAL_GEN_DIR = os.path.join(DATA_DIR, 'eval_reports')
os.makedirs(EVAL_GEN_DIR, exist_ok=True)


def count_rooms_by_type(boxes):
    counts = {t: 0 for t in ROOM_TYPES}
    for b in boxes:
        counts[b['type']] = counts.get(b['type'], 0) + 1
    return counts


def program_count_mae(requested, predicted):
    keys = set(requested.keys()) | set(predicted.keys())
    diffs = [abs(int(requested.get(k, 0)) - int(predicted.get(k, 0))) for k in keys]
    return float(np.mean(diffs)) if diffs else 0.0


def program_count_acc(requested, predicted):
    keys = set(requested.keys()) | set(predicted.keys())
    ok, total = 0, 0
    for k in keys:
        total += max(int(requested.get(k, 0)), 1)
        ok += min(int(requested.get(k, 0)), int(predicted.get(k, 0)))
    return float(ok / max(total, 1))


def modulus_compliance(boxes):
    if not boxes:
        return 0.0
    ok = 0
    total = 0
    for b in boxes:
        for k in ('box_min', 'box_max'):
            for v in b[k]:
                total += 1
                if abs(v / VOXEL_SIZE - round(v / VOXEL_SIZE)) < 1e-6:
                    ok += 1
    return ok / max(total, 1)


def site_fit_rate(boxes, site_x, site_y, site_z=9000):
    if not boxes:
        return 0.0
    ok = 0
    for b in boxes:
        bm, bx = b['box_min'], b['box_max']
        if bm[0] >= -1 and bm[1] >= -1 and bx[0] <= site_x + 1 and bx[1] <= site_y + 1 and bx[2] <= site_z + 1:
            ok += 1
    return ok / len(boxes)


def voxel_metrics_np(pred_cls, gt_cls):
    pred_cls = pred_cls.astype(np.int16)
    gt_cls = gt_cls.astype(np.int16)
    total_acc = float((pred_cls == gt_cls).mean())
    occ = gt_cls > 0
    occupied_acc = float((pred_cls[occ] == gt_cls[occ]).mean()) if occ.any() else 0.0
    ious = []
    for cid in range(1, NUM_CHANNELS):
        p, g = pred_cls == cid, gt_cls == cid
        if not g.any():
            continue
        inter = np.logical_and(p, g).sum()
        union = np.logical_or(p, g).sum()
        ious.append(inter / max(union, 1))
    return total_acc, occupied_acc, float(np.mean(ious) if ious else 0.0)


def json_to_user_request(data):
    meta = data.get('metadata', {})
    stats = meta.get('stats', {})
    bsize = meta.get('building_size', {})
    counts = {t: int(stats.get(t, 0)) for t in ROOM_TYPES if int(stats.get(t, 0)) > 0}
    return build_user_request(
        site_x=float(bsize.get('x', 18000)),
        site_y=float(bsize.get('y', 15000)),
        site_z=float(bsize.get('z', 6000)),
        room_counts=counts,
    )


@torch.no_grad()
def eval_reconstruction(sample, model):
    hg = sample['graph']
    hg.condition = sample['condition'].unsqueeze(0) if sample['condition'].dim() == 1 else sample['condition']
    batch = prepare_graph_batch(hg, condition=sample['condition']).to(device)
    logits, _, _ = forward_model(model, batch, sample['condition'])
    pred = torch.argmax(logits[0], dim=0).cpu().numpy()
    gt = torch.argmax(sample['voxel'], dim=0).numpy()
    ta, oa, miou = voxel_metrics_np(pred, gt)
    return {'total_acc': ta, 'occupied_acc': oa, 'miou': miou, 'pred': pred, 'gt': gt}


@torch.no_grad()
def eval_conditional_generation(user_req, gt_voxel, model, seed=42):
    pred, sample, rooms, G, pos = generate_voxels_from_request(user_req, model, seed=seed)
    gt = torch.argmax(gt_voxel, dim=0).numpy()
    ta, oa, miou = voxel_metrics_np(pred, gt)
    boxes = voxels_to_boxes(pred, user_req, rooms)
    pred_counts = count_rooms_by_type(boxes)
    return {
        'total_acc': ta, 'occupied_acc': oa, 'miou': miou,
        'program_mae': program_count_mae(user_req['room_counts'], pred_counts),
        'program_acc': program_count_acc(user_req['room_counts'], pred_counts),
        'modulus_rate': modulus_compliance(boxes),
        'site_fit': site_fit_rate(boxes, user_req['site_x'], user_req['site_y'], user_req['site_z']),
        'fragmentation': float(np.mean([count_components_3d(pred == cid) for cid in range(1, NUM_CHANNELS) if (pred == cid).any()]) if (pred > 0).any() else 0.0),
        'pred_counts': pred_counts,
        'boxes': boxes,
        'pred': pred,
        'gt': gt,
    }

@torch.no_grad()
def eval_conditional_generation_best_of_k(user_req, gt_voxel, model, k=None, base_seed=42):
    k = k or GEN_EVAL_K
    best = None
    best_score = (-1.0, -1.0, -1.0)
    trials = []
    for i in range(k):
        seed = base_seed + i * 17
        gen = eval_conditional_generation(user_req, gt_voxel, model, seed=seed)
        trials.append(gen)
        score = (gen['miou'], gen['program_acc'], float((gen['pred'] > 0).sum()))
        if score > best_score:
            best_score = score
            best = gen
    best = best or trials[0]
    best['eval_k'] = k
    best['trials'] = trials
    return best


print(f'Step 7a 就绪 | 条件生成 best-of-K={GEN_EVAL_K}')


In [ ]:
\
# Step 7b: 验证集批量量化评估（重建 vs 条件生成 best-of-K）
_need7 = ['model', 'eval_reconstruction', 'eval_conditional_generation_best_of_k',
          'generate_voxels_from_request', 'json_to_user_request', 'load_train_val_split']
_miss7 = [k for k in _need7 if k not in globals()]
if _miss7:
    raise RuntimeError(
        '请先运行 Step 1 → Step 1b → Step 3 → Step 7a（无需 Step 6）。\n'
        '缺少: ' + ', '.join(_miss7)
    )

set_seed(SPLIT_SEED)
wp = weight_path() if 'weight_path' in globals() else os.path.join(DATA_DIR, WEIGHT_FILENAME)
model.load_state_dict(torch.load(wp, map_location=device, weights_only=True))
model.eval()

pt_files = sorted([
    p for p in glob.glob(os.path.join(OUT_DIR, '*.pt'))
    if not os.path.basename(p).startswith('_')
])
if not pt_files:
    raise RuntimeError('请先运行 Step 2 预处理')

items = []
for fp in pt_files:
    s = torch.load(fp, weights_only=False)
    pt_name = s.get('pt_name') or os.path.basename(fp)
    items.append((pt_name, s))

all_names = [x[0] for x in items]
_, val_names, _ = load_train_val_split(all_names)
val_name_set = set(val_names)
val_items = [x for x in items if x[0] in val_name_set]
print(f'验证样本数: {len(val_items)}（来自持久化划分 {SPLIT_FILENAME}）')

rows = []
skipped = 0
for pt_name, sample in val_items:
    json_path = resolve_source_json(sample, pt_name)
    if not json_path:
        skipped += 1
        print('跳过，无 JSON:', pt_name)
        continue
    with open(json_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    user_req = json_to_user_request(raw)

    hg = sample['graph']
    hg.condition = sample['condition'].unsqueeze(0) if sample['condition'].dim() == 1 else sample['condition']
    rec_sample = {'graph': hg, 'voxel': sample['voxel'], 'condition': sample['condition']}

    rec = eval_reconstruction(rec_sample, model)
    gen = eval_conditional_generation_best_of_k(user_req, sample['voxel'], model, k=GEN_EVAL_K)

    rows.append({
        'sample': pt_name.replace('.pt', ''),
        'mode': 'reconstruction',
        'miou': rec['miou'], 'occupied_acc': rec['occupied_acc'], 'total_acc': rec['total_acc'],
        'program_mae': np.nan, 'program_acc': np.nan, 'modulus_rate': np.nan, 'site_fit': np.nan,
        'eval_k': np.nan,
    })
    rows.append({
        'sample': pt_name.replace('.pt', ''),
        'mode': 'conditional_generation_best_of_k',
        'miou': gen['miou'], 'occupied_acc': gen['occupied_acc'], 'total_acc': gen['total_acc'],
        'program_mae': gen['program_mae'], 'program_acc': gen['program_acc'],
        'modulus_rate': gen['modulus_rate'], 'site_fit': gen['site_fit'],
        'eval_k': gen.get('eval_k', GEN_EVAL_K),
    })

import pandas as pd
df_gen = pd.DataFrame(rows)
print(f'跳过（无 source_json）: {skipped}')

print('\\n=== 重建 vs 条件生成（best-of-K）对比 ===')
for mode in ['reconstruction', 'conditional_generation_best_of_k']:
    sub = df_gen[df_gen['mode'] == mode]
    if sub.empty:
        continue
    print(f"\\n[{mode}] n={len(sub)}")
    print(f"  mIoU:          {sub['miou'].mean()*100:5.2f}% ± {sub['miou'].std()*100:4.2f}%")
    print(f"  非空准确率:    {sub['occupied_acc'].mean()*100:5.2f}% ± {sub['occupied_acc'].std()*100:4.2f}%")
    if 'conditional' in mode:
        print(f"  功能数量MAE:   {sub['program_mae'].mean():5.2f} ± {sub['program_mae'].std():4.2f}")
        print(f"  功能数量匹配率:{sub['program_acc'].mean()*100:5.2f}% ± {sub['program_acc'].std()*100:4.2f}%")
        print(f"  模数合规率:    {sub['modulus_rate'].mean()*100:5.2f}%")
        print(f"  用地贴合率:    {sub['site_fit'].mean()*100:5.2f}%")
        print(f"  采样次数 K:    {sub['eval_k'].iloc[0]}")

csv_out = experiment_report_path('generation_metrics', subdir='eval_reports', ext='csv')
json_out = experiment_report_path('generation_summary', subdir='eval_reports', ext='json')
df_gen.to_csv(csv_out, index=False, encoding='utf-8-sig')

report = {}
for mode in df_gen['mode'].unique():
    sub = df_gen[df_gen['mode'] == mode]
    report[mode] = {
        c: float(sub[c].mean()) for c in [
            'miou', 'occupied_acc', 'total_acc', 'program_mae', 'program_acc', 'modulus_rate', 'site_fit'
        ] if c in sub.columns and sub[c].notna().any()
    }
report['eval_k'] = GEN_EVAL_K
with open(json_out, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"\\n已导出: {csv_out}")
print(f"已导出: {json_out}")
try:
    display(df_gen.pivot_table(index='sample', columns='mode', values=['miou', 'occupied_acc', 'program_acc']))
except NameError:
    print(df_gen.head(10))


## Step 8: 交互式条件生成（Colab 推荐：ipywidgets）

- **显示**：`plot_3d_layout` + `plot_2d_floorplan` 静态图，`show_layout_fig((fig3d, fig2d))`。
- **`非空体素=0`**：模型对「合成拓扑」解码失败，界面会 **自动用布局引擎体块兜底**（仍应能看到 3D）。
- **Gradio**：可选，导出 PNG 预览；日常请用本 Step。


In [ ]:
# Step 8: Colab 交互生成（ipywidgets + matplotlib 3D，推荐）
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
ensure_model_loaded(force_rebuild_model=False)

w_site_x = widgets.IntSlider(value=18000, min=6000, max=30000, step=300, description='用地X', style={'description_width': '60px'}, layout=widgets.Layout(width='420px'))
w_site_y = widgets.IntSlider(value=15000, min=6000, max=30000, step=300, description='用地Y', style={'description_width': '60px'}, layout=widgets.Layout(width='420px'))
w_seed = widgets.IntText(value=0, description='种子(0=随机)', style={'description_width': '80px'})

fields = [
    ('entryway', '玄关', 1), ('living_room', '客厅', 1), ('dining_room', '餐厅', 1), ('kitchen', '厨房', 1),
    ('bedroom', '卧室', 3), ('bathroom', '卫生间', 2), ('corridor', '过道', 2),
    ('stairs', '楼梯', 1), ('utility', '储藏', 0), ('balcony', '阳台', 1), ('multi_purpose', '多功能', 0),
]
room_widgets = {}
rows = []
for i in range(0, len(fields), 4):
    row = []
    for key, label, default in fields[i:i+4]:
        w = widgets.IntText(value=default, description=label, style={'description_width': '50px'}, layout=widgets.Layout(width='140px'))
        room_widgets[key] = w
        row.append(w)
    rows.append(widgets.HBox(row))

btn = widgets.Button(description='生成三维体块', button_style='primary', icon='cube')
text_out = widgets.Output()
plot_out = widgets.Output(layout=widgets.Layout(min_height='760px', border='1px solid #ddd'))


def on_generate(_=None):
    req = None
    result = None
    plot_rooms = []
    src = '未知'
    try:
        counts = {k: int(w.value) for k, w in room_widgets.items() if int(w.value) > 0}
        req = build_user_request(w_site_x.value, w_site_y.value, counts)
        seed = int(w_seed.value) if int(w_seed.value) > 0 else None
        result = generate_user_layout(req, model, seed=seed)
        plot_rooms = result.get('display_rooms', result['rooms'])
        boxes = rooms_to_boxes(plot_rooms)
        pred_counts = count_rooms_by_type(boxes)
        src = '模型体素' if result.get('display_source') == 'model_voxels' else '布局引擎兜底'

        with text_out:
            clear_output(wait=True)
            print(f"seed={result['seed']} | 解码={result['decode_mode']} | 非空体素={result['n_occ']} | 显示={src}")
            if result['n_occ'] == 0:
                print('提示: 模型体素为空（合成拓扑与训练数据分布不一致），已用布局引擎体块显示。可固定种子 123 多试几次。')
            print(f"请求: {req['room_counts']}")
            print(f"布局: {pred_counts}")
            print(f"功能MAE={program_count_mae(req['room_counts'], pred_counts):.2f} | "
                  f"匹配率={program_count_acc(req['room_counts'], pred_counts)*100:.1f}% | "
                  f"模数={modulus_compliance(boxes)*100:.1f}% | 用地={site_fit_rate(boxes, req['site_x'], req['site_y'])*100:.1f}%")

        with plot_out:
            clear_output(wait=True)
            _pk = dict(
                graph=result.get('graph'), edge_types=result.get('edge_types'), layout_rooms=result.get('rooms'),
            )
            fig3d = plot_3d_layout(
                plot_rooms, req['site_x'], req['site_y'],
                title=f"3D 功能体块 (seed={result['seed']}, {src})",
                **_pk,
            )
            fig2d = plot_2d_floorplan(
                plot_rooms, req['site_x'], req['site_y'],
                title=f"平面图 (seed={result['seed']})",
                **_pk,
            )
            show_layout_fig((fig3d, fig2d))
    except Exception as e:
        with text_out:
            clear_output(wait=True)
            print(f'生成失败: {e}')
            print('请先运行 Step 1 → Step 1b → Step 3 → Step 6a，再重跑 Step 8。')

btn.on_click(on_generate)

display(widgets.VBox([
    widgets.HTML('<b>输入用地与功能需求 → 点击下方按钮生成（可反复点击，输入不会清空）</b>'),
    widgets.HBox([w_site_x, w_site_y, w_seed]),
    *rows,
    btn,
    text_out,
    plot_out,
]))
print('Step 8 就绪：文字在上、matplotlib 3D 在下；请先运行 Step 6a 再重跑本 cell')


### Step 8-可选：Gradio 网页 UI（需对外分享时用）

注意：`debug=True` 会导致 Colab **反复重跑单元格**；已改为 `debug=False`。
3D 预览在 Colab 内仍建议用上方 **Step 8 ipywidgets**。


In [ ]:
# Step 8-可选: Gradio（debug=False 避免自动重跑）
# !pip -q install gradio

import gradio as gr

ensure_model_loaded(force_rebuild_model=False)


def gradio_generate(site_x, site_y, seed_input, n_entry, n_living, n_dining, n_kitchen,
                    n_bed, n_bath, n_corr, n_stairs, n_util, n_balc, n_multi):
    req = build_user_request(site_x, site_y, {
        'entryway': int(n_entry), 'living_room': int(n_living), 'dining_room': int(n_dining),
        'kitchen': int(n_kitchen), 'bedroom': int(n_bed), 'bathroom': int(n_bath),
        'corridor': int(n_corr), 'stairs': int(n_stairs), 'utility': int(n_util),
        'balcony': int(n_balc), 'multi_purpose': int(n_multi),
    })
    seed = int(seed_input) if int(seed_input) > 0 else None
    out = generate_user_layout(req, model, seed=seed)
    boxes = rooms_to_boxes(out['rooms'])
    fig = plot_3d_layout(out['rooms'], site_x, site_y, title=f'seed={out["seed"]}')
    html = fig.to_html(full_html=False, include_plotlyjs='cdn')
    metrics = (
        f"seed={out['seed']}\n非空体素={out['n_occ']}\n解码={out['decode_mode']}\n"
        f"匹配率={program_count_acc(req['room_counts'], count_rooms_by_type(boxes))*100:.1f}%"
    )
    return html, metrics


with gr.Blocks() as demo:
    gr.Markdown('## Gradio 条件生成（可选）')
    with gr.Row():
        site_x = gr.Slider(6000, 30000, value=18000, step=300, label='X')
        site_y = gr.Slider(6000, 30000, value=15000, step=300, label='Y')
        seed_in = gr.Number(0, label='种子', precision=0)
    with gr.Row():
        n_entry = gr.Number(1, label='玄关', precision=0)
        n_living = gr.Number(1, label='客厅', precision=0)
        n_dining = gr.Number(1, label='餐厅', precision=0)
        n_kitchen = gr.Number(1, label='厨房', precision=0)
    with gr.Row():
        n_bed = gr.Number(3, label='卧室', precision=0)
        n_bath = gr.Number(2, label='卫生间', precision=0)
        n_corr = gr.Number(2, label='过道', precision=0)
    with gr.Row():
        n_stairs = gr.Number(1, label='楼梯', precision=0)
        n_util = gr.Number(0, label='储藏', precision=0)
        n_balc = gr.Number(1, label='阳台', precision=0)
        n_multi = gr.Number(0, label='多功能', precision=0)
    btn = gr.Button('生成')
    plot_html = gr.HTML(label='3D HTML')
    metrics = gr.Textbox(label='指标')
    btn.click(gradio_generate,
              inputs=[site_x, site_y, seed_in, n_entry, n_living, n_dining, n_kitchen,
                      n_bed, n_bath, n_corr, n_stairs, n_util, n_balc, n_multi],
              outputs=[plot_html, metrics])

demo.launch(debug=False, share=True)
